## 初期設定

In [94]:
# 基本モジュール
import sys, os
from IPython.display import display, HTML, clear_output, update_display, Image
import numpy as np
import matplotlib.pyplot as plt
import pyFAI
import pandas as pd
import h5py

# 自作モジュール
sys.path.append(r"C:\Users\okaza\pythonenv")
from modules.Mytools.handle_ipynb import save_pickle, load_pickle, export_html, ask_openfilename, ask_savefilename
from modules.Mytools.Tools import h5_tree, dict_tree, his2array
sys.path.append(os.getcwd())
from main import Create_Hdfdata # type: ignore

# 初期パラメーター
cachedir = os.path.join(os.getcwd(), ".cache")
os.makedirs(cachedir, exist_ok=True)

## 初期化

In [52]:
Experiment = "UODE36_0013"

In [ ]:
# Create_Hdfdataのインスタント化
i2a = Create_Hdfdata(
    name = Experiment,
    cachedir = cachedir,
    log = True
)

name: UODE36_0013
cachedir: c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\.cache
log: True
version: 1.0


## ファイルリストの作成

In [51]:
# ディレクトリ名を指定
dir = r"D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1"

# ヘッダーとフッターを指定
header = "UODE36_13_"
footer = ".his"

In [ ]:
# ファイルリストの格納
## ヘッダーとフッターを含むファイル名を取得
flist = list()
for __ in os.listdir(dir):
    if not header in __:
        continue
    if not footer in __:
        continue
    flist.append(__)

## ソート
flist.sort(key = (lambda x: int(x.replace(header, "").replace(footer, ""))))
filenames = list(map(lambda x: os.path.join(dir, x), flist))

## 表示
for f in filenames:
    print(f)

## 格納
i2a.filelist = filenames

D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_0.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_1.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_2.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_3.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_4.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_5.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_6.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_7.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_8.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_9.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_10.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_11.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_12.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_13.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UODE36_13_14.his
D:\DATA\SPring-8-2025-Dec\Okazaki\UODE36\FPD_1\UOD

## 一次元化

In [19]:
# 校正用ファイル
poni = r"D:\DATA\SPring-8-2025-Dec\ceo2_20251128_EH2_NB\IPAnalyzer_IP_MgS400_EH2_20251215_YusukeOkazaki.poni"

# 細かさ
npt = 3000

# 2thetaの範囲
radial_range = (3,30)

In [ ]:
# poniファイルやradial_rangeを確認する
def comfirm_rad_range():

    # 表示するフレーム
    frame = 4

    # 強度の幅
    intensity_range = (10,200)
    
    # データの読み込み
    hisfile = i2a.filelist[frame]
    hisdata = his2array(hisfile)
    hist, bins =  np.histogram(hisdata, bins = np.linspace(*intensity_range, 100))
    ai = pyFAI.load(poni)
    data_1d = ai.integrate1d(
        hisdata,
        npt = npt,
        unit = "2th_deg",
        method = "ocl",
        radial_range=radial_range
    )

    # figure作成
    fig = plt.figure()
    fig.set_size_inches(9,6)
    ax_imshow = fig.add_axes(rect = (0.02,0.2,0.45,0.6))
    ax_hist = fig.add_axes(rect = (0.55,0.6,0.4,0.3))
    ax_profile = fig.add_axes(rect = (0.55,0.1,0.4,0.3))

    ax_imshow.set_xticks([])
    ax_imshow.set_yticks([])

    ax_hist.stairs(values = hist, edges = bins, lw = 1,  color = "0")
    ax_hist.set_xlabel("Intensity")
    ax_hist.set_ylabel("Count")
    
    ax_imshow.imshow(hisdata,vmin = intensity_range[0], vmax = intensity_range[1])
    ax_imshow.set_title(os.path.splitext(os.path.basename(hisfile))[0])

    ax_profile.plot(*data_1d, lw = 1, c = "0")
    ax_profile.set_xlabel("2theta [deg]")
    ax_profile.set_ylabel("Intensity")

    imgfilename = os.path.join(cachedir, "comfirm_rad_range.png")
    plt.savefig(imgfilename, dpi = 300)
    plt.close()
    print(os.path.abspath(imgfilename))

    return
comfirm_rad_range()
del comfirm_rad_range

c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\.cache\comfirm_rad_range.png


In [ ]:
# 一気に1次元化する
hdf_1d = i2a.integrate1D(
    poni = poni,
    npt_rad = npt,
    radial_range = radial_range,
)

100%|██████████| 136/136 [00:00<00:00, 30564.50it/s]


Progress: [■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■] 100% (136/136) 

Integration completed.
<HDF5 file "UODE36_0013_integrated.hdf" (mode r)>
├── integrated
│   ├── frame = 0 ((3000,), float32)
│   ├── frame = 1 ((3000,), float32)
│   ├── frame = 10 ((3000,), float32)
│   ├── frame = 100 ((3000,), float32)
│   ├── frame = 101 ((3000,), float32)
│   ├── frame = 102 ((3000,), float32)
│   ├── frame = 103 ((3000,), float32)
│   ├── frame = 104 ((3000,), float32)
│   ├── frame = 105 ((3000,), float32)
│   ├── frame = 106 ((3000,), float32)
│   ├── frame = 107 ((3000,), float32)
│   ├── frame = 108 ((3000,), float32)
│   ├── frame = 109 ((3000,), float32)
│   ├── frame = 11 ((3000,), float32)
│   ├── frame = 110 ((3000,), float32)
│   ├── frame = 111 ((3000,), float32)
│   ├── frame = 112 ((3000,), float32)
│   ├── frame = 113 ((3000,), float32)
│   ├── frame = 114 ((3000,), float32)
│   ├── frame = 115 ((3000,), float32)
│   ├── frame = 116 ((3000,), float32)
│   ├── frame = 117 ((3000,),

In [ ]:
# 一次元化の結果を表示する（matplotlib）
def disp_caking():

    # 表示する角度幅
    rad_range = (13,14)

    # データ読み込み
    d = []
    with h5py.File(hdf_1d, mode = "r") as f:
        tth = np.array(f["rad"][()]) # type: ignore

        mask = (tth > rad_range[0])&(tth < rad_range[1])
        tth = tth[mask]

        n_frame = len(f["integrated"].keys()) # type: ignore
        for i in range(n_frame):
            d.append(np.array(f["integrated/frame = {}".format(i)][()])[mask]) # type: ignore

    # imshow
    fig, axs = plt.subplots(2,1)
    fig.set_size_inches(6,9)
    axs[0].imshow(d[::-1],
               extent = (tth[0], tth[-1], 0, n_frame),
               aspect = "auto",
               )

    # plot
    aset = 5
    for i in range(n_frame):
        axs[1].plot(tth, d[i]+aset*i, lw = 0.1)
        axs[1].set_xlim(rad_range)

    # 出力
    imgfilename = os.path.join(cachedir, "disp_caking.png")
    plt.savefig(imgfilename, dpi = 300)
    plt.close()
    print(imgfilename)
    

    return
disp_caking()
del disp_caking

c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\.cache\disp_caking.png


In [112]:
# 一次元化の結果を表示する（plotly）
## http://127.0.0.1:8000/ にアクセスしてください
!python "C:\Users\okaza\Documents\Documents\fpd\dash_Integrated1D\Dash_Integrated1D.py"

^C


In [ ]:
# PDIndexer用にcsvを保存する
def save_csv():

    # 1枚をcsvにするか、複数のフレームの平均をcsvにするか
    ## flag_1shot = True
    if not "flag_1shot" in locals():
        flag_1shot = False

    ## 1枚の場合
    if flag_1shot:
        frame = 0
        with h5py.File(hdf_1d, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore
            insty = np.array(f["integrated/frame = {}".format(frame)][()]) # type: ignore
        csvfilename = os.path.join(cachedir, os.path.basename(os.path.splitext(i2a.filelist[frame])[0]) + ".csv")

    ## 複数フレーム
    else:
        frame_range = (0,22)
        with h5py.File(hdf_1d, mode = "r") as f:
            tth = np.array(f["rad"][()]) # type: ignore

            insty = np.zeros(shape = tth.shape)
            for i in range(*frame_range):
                insty += np.array(f["integrated/frame = {}".format(i)][()]) # type: ignore
            insty /= (frame_range[1] - frame_range[0])
            
        csvfilename = os.path.join(cachedir, header + "{}-{}.csv".format(*frame_range))

    ## 保存
    df = pd.DataFrame([tth, insty]).T
    df.to_csv(csvfilename, index = False, header = False)
    print(csvfilename)
    display(df)

    return
save_csv()
del save_csv

c:\Users\okaza\Documents\Documents\fpd\create_hdfdata\.cache\UODE36_13_0-22.csv


,0,1
0,3.004500,29.750858
1,3.013500,29.583287
2,3.022500,29.834003
3,3.031500,29.722618
4,3.040500,29.543637
...,...,...
2995,29.959505,61.510682
2996,29.968504,61.041046
2997,29.977505,60.616321
2998,29.986504,60.184732


## Unroll

In [115]:
npt_rad = 200
npt_azim = 720
radial_range = (12,14)

In [ ]:
# unrollを行う
hdf_2d = i2a.integrate2D(
    poni = poni,
    npt_rad = npt_rad,
    npt_azim = npt_azim,
    radial_range = radial_range,
)

100%|██████████| 136/136 [00:00<00:00, 28202.58it/s]


Progress: [■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■■] 100% (136/136) 

Integration completed.
<HDF5 file "integrated.hdf" (mode r)>
├── azim ((720,), float32)
├── integrated
│   ├── frame = 0 ((720, 200), float32)
│   ├── frame = 1 ((720, 200), float32)
│   ├── frame = 10 ((720, 200), float32)
│   ├── frame = 100 ((720, 200), float32)
│   ├── frame = 101 ((720, 200), float32)
│   ├── frame = 102 ((720, 200), float32)
│   ├── frame = 103 ((720, 200), float32)
│   ├── frame = 104 ((720, 200), float32)
│   ├── frame = 105 ((720, 200), float32)
│   ├── frame = 106 ((720, 200), float32)
│   ├── frame = 107 ((720, 200), float32)
│   ├── frame = 108 ((720, 200), float32)
│   ├── frame = 109 ((720, 200), float32)
│   ├── frame = 11 ((720, 200), float32)
│   ├── frame = 110 ((720, 200), float32)
│   ├── frame = 111 ((720, 200), float32)
│   ├── frame = 112 ((720, 200), float32)
│   ├── frame = 113 ((720, 200), float32)
│   ├── frame = 114 ((720, 200), float32)
│   ├── frame = 115 ((720, 200), fl

In [120]:
# unrollの結果を表示する（plotly）
## http://127.0.0.1:8025/ にアクセスしてください
!python "C:\Users\okaza\Documents\Documents\fpd\dash_Integrated2D\Dash_Integrated2D.py"

^C
